# آستانه‌گذاری تصویر / Image Thresholding

**هدف:** یادگیری و پیاده‌سازی تکنیک‌های مختلف آستانه‌گذاری برای تبدیل تصاویر خاکستری به تصاویر دودویی

**Objective:** Learn and implement various thresholding techniques to convert grayscale images to binary images

---

## محتوا / Contents:
1. آستانه‌گذاری ساده / Simple Thresholding
2. انواع آستانه‌گذاری / Threshold Types
3. روش اتسو / Otsu's Method
4. آستانه‌گذاری تطبیقی / Adaptive Thresholding
5. مقایسه روش‌ها / Comparison of Methods

In [ ]:
# Import libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple

# تنظیمات نمایش / Display settings
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 12

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. توابع کمکی / Helper Functions

In [ ]:
def add_gaussian_noise(image: np.ndarray, mean: float = 0, sigma: float = 20) -> np.ndarray:
    """
    اضافه کردن نویز گاوسی به تصویر
    
    Args:
        image: تصویر ورودی
        mean: میانگین نویز (معمولاً 0)
        sigma: انحراف معیار نویز
    
    Returns:
        تصویر نویزی
    """
    # تبدیل به int16 برای جلوگیری از overflow
    image_int16 = image.astype(np.int16)
    
    # تولید نویز گاوسی
    noise = np.random.normal(mean, sigma, image.shape).astype(np.int16)
    
    # اضافه کردن نویز
    noisy_image = image_int16 + noise
    
    # محدود کردن به بازه [0, 255]
    noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)
    
    return noisy_image


def calculate_snr(original: np.ndarray, noisy: np.ndarray) -> float:
    """
    محاسبه نسبت سیگنال به نویز (SNR)
    
    Args:
        original: تصویر اصلی
        noisy: تصویر نویزی
    
    Returns:
        SNR به دسی‌بل
    """
    signal_power = np.sum(original.astype(np.float64) ** 2)
    noise = (noisy.astype(np.float64) - original.astype(np.float64))
    noise_power = np.sum(noise ** 2)
    
    if noise_power == 0:
        return float('inf')
    
    snr = 10 * np.log10(signal_power / noise_power)
    return snr


def display_images(images: list, titles: list, cmap='gray'):
    """
    نمایش چند تصویر در کنار هم
    """
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
    
    if n == 1:
        axes = [axes]
    
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

print("✓ توابع کمکی آماده شدند")

## 2. ایجاد تصویر نمونه / Create Sample Image

In [ ]:
# ایجاد تصویر نمونه با اشکال مختلف
# Create sample image with different shapes
image = np.zeros((400, 600), dtype=np.uint8)

# اضافه کردن مستطیل‌ها با روشنایی‌های مختلف
# Add rectangles with different intensities
cv2.rectangle(image, (50, 50), (150, 150), 80, -1)
cv2.rectangle(image, (200, 50), (300, 150), 150, -1)
cv2.rectangle(image, (350, 50), (450, 150), 220, -1)

# اضافه کردن دایره‌ها
# Add circles
cv2.circle(image, (100, 250), 50, 100, -1)
cv2.circle(image, (250, 250), 50, 180, -1)
cv2.circle(image, (400, 250), 50, 240, -1)

# اضافه کردن نویز
# Add noise
noisy_image = add_gaussian_noise(image, sigma=15)

display_images([image, noisy_image], 
               ['تصویر اصلی / Original', 'تصویر نویزی / Noisy'])

print(f"اندازه تصویر / Image size: {image.shape}")
print(f"SNR: {calculate_snr(image, noisy_image):.2f} dB")

## 3. آستانه‌گذاری ساده / Simple Thresholding

در آستانه‌گذاری ساده، یک مقدار آستانه ثابت برای کل تصویر استفاده می‌شود.

In simple thresholding, a fixed threshold value is used for the entire image.

In [ ]:
# آستانه‌گذاری با مقادیر مختلف
# Thresholding with different values
threshold_values = [100, 150, 200]
results = []
titles = ['تصویر اصلی\nOriginal']

for thresh in threshold_values:
    # اعمال آستانه‌گذاری دودویی
    # Apply binary thresholding
    ret, binary = cv2.threshold(noisy_image, thresh, 255, cv2.THRESH_BINARY)
    results.append(binary)
    titles.append(f'آستانه = {thresh}\nThreshold = {thresh}')

# نمایش نتایج
# Display results
all_images = [noisy_image] + results
display_images(all_images, titles)

print("✓ آستانه‌گذاری ساده انجام شد")

## 4. انواع آستانه‌گذاری / Threshold Types

OpenCV انواع مختلفی از آستانه‌گذاری را ارائه می‌دهد:

OpenCV provides different types of thresholding:

- **BINARY**: پیکسل‌های بالاتر از آستانه = 255، بقیه = 0
- **BINARY_INV**: معکوس BINARY
- **TRUNC**: پیکسل‌های بالاتر از آستانه = آستانه، بقیه بدون تغییر
- **TOZERO**: پیکسل‌های پایین‌تر از آستانه = 0، بقیه بدون تغییر
- **TOZERO_INV**: معکوس TOZERO

In [ ]:
# آستانه ثابت برای همه روش‌ها
# Fixed threshold for all methods
thresh_value = 150

# اعمال انواع مختلف آستانه‌گذاری
# Apply different threshold types
ret, binary = cv2.threshold(noisy_image, thresh_value, 255, cv2.THRESH_BINARY)
ret, binary_inv = cv2.threshold(noisy_image, thresh_value, 255, cv2.THRESH_BINARY_INV)
ret, trunc = cv2.threshold(noisy_image, thresh_value, 255, cv2.THRESH_TRUNC)
ret, tozero = cv2.threshold(noisy_image, thresh_value, 255, cv2.THRESH_TOZERO)
ret, tozero_inv = cv2.threshold(noisy_image, thresh_value, 255, cv2.THRESH_TOZERO_INV)

# نمایش نتایج
# Display results
images = [noisy_image, binary, binary_inv, trunc, tozero, tozero_inv]
titles = ['اصلی\nOriginal', 'BINARY', 'BINARY_INV', 'TRUNC', 'TOZERO', 'TOZERO_INV']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

print("✓ انواع آستانه‌گذاری نمایش داده شد")

## 5. روش اتسو / Otsu's Method

روش اتسو به‌طور خودکار بهترین مقدار آستانه را محاسبه می‌کند.

Otsu's method automatically calculates the optimal threshold value.

In [ ]:
# آستانه‌گذاری با روش اتسو
# Thresholding with Otsu's method
ret_otsu, otsu = cv2.threshold(noisy_image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# آستانه‌گذاری دستی با همان مقدار
# Manual thresholding with same value
ret_manual, manual = cv2.threshold(noisy_image, ret_otsu, 255, cv2.THRESH_BINARY)

# نمایش هیستوگرام و آستانه اتسو
# Display histogram and Otsu threshold
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# تصویر اصلی
axes[0, 0].imshow(noisy_image, cmap='gray')
axes[0, 0].set_title('تصویر نویزی\nNoisy Image', fontweight='bold')
axes[0, 0].axis('off')

# هیستوگرام
axes[0, 1].hist(noisy_image.ravel(), 256, [0, 256])
axes[0, 1].axvline(ret_otsu, color='r', linestyle='--', linewidth=2, label=f'Otsu = {ret_otsu:.0f}')
axes[0, 1].set_title('هیستوگرام\nHistogram', fontweight='bold')
axes[0, 1].set_xlabel('مقدار پیکسل / Pixel Value')
axes[0, 1].set_ylabel('تعداد / Count')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# نتیجه اتسو
axes[1, 0].imshow(otsu, cmap='gray')
axes[1, 0].set_title(f'روش اتسو (آستانه={ret_otsu:.0f})\nOtsu Method', fontweight='bold')
axes[1, 0].axis('off')

# مقایسه با آستانه دستی
axes[1, 1].imshow(manual, cmap='gray')
axes[1, 1].set_title(f'آستانه دستی (آستانه={ret_otsu:.0f})\nManual Threshold', fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print(f"آستانه بهینه اتسو / Optimal Otsu threshold: {ret_otsu:.2f}")

## 6. آستانه‌گذاری تطبیقی / Adaptive Thresholding

در آستانه‌گذاری تطبیقی، آستانه برای هر ناحیه از تصویر به‌طور جداگانه محاسبه می‌شود.

In adaptive thresholding, the threshold is calculated separately for each region of the image.

In [ ]:
# ایجاد تصویر با نور نامتجانس
# Create image with non-uniform illumination
gradient_image = np.zeros((400, 600), dtype=np.uint8)

# اضافه کردن گرادیان نور
# Add illumination gradient
for i in range(400):
    for j in range(600):
        gradient_image[i, j] = int(50 + (j / 600) * 150)

# اضافه کردن اشکال
# Add shapes
cv2.rectangle(gradient_image, (50, 50), (150, 150), 255, -1)
cv2.rectangle(gradient_image, (200, 50), (300, 150), 255, -1)
cv2.rectangle(gradient_image, (350, 50), (450, 150), 255, -1)
cv2.circle(gradient_image, (100, 250), 50, 255, -1)
cv2.circle(gradient_image, (250, 250), 50, 255, -1)
cv2.circle(gradient_image, (400, 250), 50, 255, -1)

# اضافه کردن نویز
# Add noise
gradient_noisy = add_gaussian_noise(gradient_image, sigma=10)

# آستانه‌گذاری سراسری (اتسو)
# Global thresholding (Otsu)
ret, global_thresh = cv2.threshold(gradient_noisy, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# آستانه‌گذاری تطبیقی - میانگین
# Adaptive thresholding - Mean
adaptive_mean = cv2.adaptiveThreshold(gradient_noisy, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                      cv2.THRESH_BINARY, 21, 10)

# آستانه‌گذاری تطبیقی - گاوسی
# Adaptive thresholding - Gaussian
adaptive_gaussian = cv2.adaptiveThreshold(gradient_noisy, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                          cv2.THRESH_BINARY, 21, 10)

# نمایش نتایج
# Display results
images = [gradient_noisy, global_thresh, adaptive_mean, adaptive_gaussian]
titles = ['تصویر با نور نامتجانس\nNon-uniform Illumination',
          f'آستانه سراسری (اتسو={ret:.0f})\nGlobal (Otsu)',
          'آستانه تطبیقی (میانگین)\nAdaptive (Mean)',
          'آستانه تطبیقی (گاوسی)\nAdaptive (Gaussian)']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

print("✓ آستانه‌گذاری تطبیقی انجام شد")

## 7. مقایسه روش‌ها / Comparison of Methods

مقایسه عملکرد روش‌های مختلف آستانه‌گذاری

Comparing the performance of different thresholding methods

In [ ]:
# تست روی تصویر واقعی (در صورت وجود)
# Test on real image (if available)
import os

# سعی در خواندن تصویر از فصل‌های قبلی
# Try to read image from previous chapters
test_image_path = '../chapter0_opencv_tutorial/checkerboard_84x84.jpg'

if os.path.exists(test_image_path):
    test_image = cv2.imread(test_image_path, cv2.IMREAD_GRAYSCALE)
    
    # اعمال روش‌های مختلف
    # Apply different methods
    ret_simple, simple = cv2.threshold(test_image, 127, 255, cv2.THRESH_BINARY)
    ret_otsu, otsu = cv2.threshold(test_image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    adaptive_mean = cv2.adaptiveThreshold(test_image, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                          cv2.THRESH_BINARY, 11, 2)
    adaptive_gaussian = cv2.adaptiveThreshold(test_image, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                              cv2.THRESH_BINARY, 11, 2)
    
    # نمایش نتایج
    # Display results
    images = [test_image, simple, otsu, adaptive_mean, adaptive_gaussian]
    titles = ['اصلی\nOriginal', 
              f'ساده (127)\nSimple', 
              f'اتسو ({ret_otsu:.0f})\nOtsu',
              'تطبیقی میانگین\nAdaptive Mean',
              'تطبیقی گاوسی\nAdaptive Gaussian']
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for i, (ax, img, title) in enumerate(zip(axes, images, titles)):
        ax.imshow(img, cmap='gray')
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.axis('off')
    
    # حذف محور اضافی
    axes[-1].remove()
    
    plt.tight_layout()
    plt.show()
    
    print("✓ مقایسه روش‌ها انجام شد")
else:
    print("⚠ تصویر تست یافت نشد")

## 8. تمرین‌ها / Exercises

### تمرین 1 / Exercise 1
تصویری با نور نامتجانس ایجاد کنید و روش‌های مختلف آستانه‌گذاری را روی آن امتحان کنید.

Create an image with non-uniform illumination and test different thresholding methods on it.

### تمرین 2 / Exercise 2
پارامترهای آستانه‌گذاری تطبیقی (اندازه بلوک و ثابت C) را تغییر دهید و تأثیر آن‌ها را بررسی کنید.

Change the adaptive thresholding parameters (block size and constant C) and examine their effects.

### تمرین 3 / Exercise 3
تصویری با نویز زیاد ایجاد کنید و ببینید کدام روش آستانه‌گذاری بهتر عمل می‌کند.

Create an image with high noise and see which thresholding method performs better.